# ML Pipeline 3: Social Media Post Performance & Donation Conversion

## Problem Framing

**Business Question**: What content actually drives donations? Which post characteristics (topic, sentiment, timing, platform) correlate with donation revenue?

**Why It Matters**: The case states: "Social media is the organization's primary channel... but they freely admit they're not experienced with social media. What kind of content actually leads to donations versus just generating likes?"

**Modeling Approach**:
- **Predictive**: Regression model to predict donation revenue from a social media post
- **Explanatory**: Which post characteristics drive donations? (post_type, sentiment, media_type, has_call_to_action, platform, timing)
- **Optimization**: Recommend best post characteristics for revenue

**Success Metrics**:
- Predictive: RMSE < â‚±50K on post revenue prediction
- Explanatory: Identify top 5 content features that drive donations
- Operational: Recommend post strategy (post types, timing, content topics)


In [ ]:
import sys
import pathlib
import warnings
warnings.filterwarnings('ignore')

# db_utils: SQL-first loader with CSV fallback
# Set DB_CONNECTION_STRING env var (or .env file in ml-pipelines/) to use Azure SQL.
# Automatically falls back to the local CSV files if SQL is unavailable.
sys.path.insert(0, str(pathlib.Path().resolve()))
import db_utils as db
print(f'Data source: {db.connection_status()}')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     KFold, GridSearchCV)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load data (SQL first, CSV fallback)
posts, donations = db.load_tables('social_media_posts', 'donations')
donations['donation_date'] = pd.to_datetime(donations['donation_date'])
print(f'Posts:     {posts.shape}')
print(f'Donations: {donations.shape}')


## Data Preparation & Feature Engineering

In [ ]:
# Target: Donation revenue from post
df = posts[[
    'post_id', 'platform', 'post_type', 'media_type', 'sentiment_tone', 'content_topic',
    'has_call_to_action', 'call_to_action_type', 'features_resident_story', 'is_boosted',
    'boost_budget_php', 'day_of_week', 'post_hour', 'caption_length', 'num_hashtags',
    'impressions', 'reach', 'engagement_rate', 'likes', 'comments', 'shares',
    'estimated_donation_value_php', 'donation_referrals'
]].copy()

# Fill missing values
df['estimated_donation_value_php'] = df['estimated_donation_value_php'].fillna(0)
df['boost_budget_php'] = df['boost_budget_php'].fillna(0)
df['is_boosted'] = df['is_boosted'].fillna(0).astype(int)
df['donation_referrals'] = df['donation_referrals'].fillna(0)
df['features_resident_story'] = df['features_resident_story'].fillna(0).astype(int)
df['has_call_to_action'] = df['has_call_to_action'].fillna(0).astype(int)

# Filter to posts with engagement data
df = df[df['impressions'] > 0].copy()

# Create target: log of donation value (to normalize skewed distribution)
df['log_donation_value'] = np.log1p(df['estimated_donation_value_php'])

print(f"Posts with engagement: {len(df)}")
print(f"Posts with donations: {(df['estimated_donation_value_php'] > 0).sum()}")
print(f"\nDonation value statistics:")
print(df['estimated_donation_value_php'].describe())

## Exploration: What Content Drives Donations?

In [ ]:
# Which post types generate donations?
print("Average donation revenue by post type:")
print(df.groupby('post_type')['estimated_donation_value_php'].agg(['count', 'mean', 'sum']).sort_values('mean', ascending=False))

print("\nAverage donation revenue by sentiment:")
print(df.groupby('sentiment_tone')['estimated_donation_value_php'].agg(['count', 'mean', 'sum']).sort_values('mean', ascending=False))

print("\nAverage donation revenue by media type:")
print(df.groupby('media_type')['estimated_donation_value_php'].agg(['count', 'mean', 'sum']).sort_values('mean', ascending=False))

print("\nAverage donation revenue by platform:")
print(df.groupby('platform')['estimated_donation_value_php'].agg(['count', 'mean', 'sum']).sort_values('mean', ascending=False))

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

df.boxplot(column='estimated_donation_value_php', by='post_type', ax=axes[0, 0])
axes[0, 0].set_title('Donation by Post Type')
axes[0, 0].set_ylabel('Donation Value (â‚±)')

df.boxplot(column='estimated_donation_value_php', by='sentiment_tone', ax=axes[0, 1])
axes[0, 1].set_title('Donation by Sentiment')
axes[0, 1].set_ylabel('Donation Value (â‚±)')

df.boxplot(column='estimated_donation_value_php', by='media_type', ax=axes[1, 0])
axes[1, 0].set_title('Donation by Media Type')
axes[1, 0].set_ylabel('Donation Value (â‚±)')

df.boxplot(column='estimated_donation_value_php', by='platform', ax=axes[1, 1])
axes[1, 1].set_title('Donation by Platform')
axes[1, 1].set_ylabel('Donation Value (â‚±)')

plt.tight_layout()
plt.show()

## Modeling & Feature Importance

In [ ]:
# Build feature matrix
feature_cols = [
    'has_call_to_action', 'features_resident_story', 'is_boosted', 'boost_budget_php',
    'post_hour', 'caption_length', 'num_hashtags', 'impressions', 'reach', 
    'engagement_rate', 'likes', 'comments', 'shares'
]

cat_cols = ['platform', 'post_type', 'media_type', 'sentiment_tone', 'content_topic', 'day_of_week']

X = df[feature_cols + cat_cols].copy()
y = df['estimated_donation_value_php'].copy()

# Encode categoricals
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].fillna('Unknown'))

X = X.fillna(X.median(numeric_only=True))

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Models
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = {}
predictions = {}

for name, model in models.items():
    if name == 'Linear Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    predictions[name] = (model, y_pred)
    
    print(f"{name}:")
    print(f"  RMSE: â‚±{rmse:.2f}")
    print(f"  MAE:  â‚±{mae:.2f}")
    print(f"  RÂ²:   {r2:.4f}")
    print()

In [ ]:
### Explanatory Model: Linear Regression Coefficients
# Linear Regression gives interpretable coefficients — shows the *direction and magnitude*
# of each feature's relationship with donation revenue (pesos per unit change).

lr_model = models['Linear Regression']
lr_model.fit(X_train_scaled, y_train)

coef_df = pd.DataFrame({
    'feature': feature_cols + cat_cols,
    'coefficient': lr_model.coef_
}).sort_values('coefficient', key=abs, ascending=False)

print("=== EXPLANATORY MODEL: Linear Regression Coefficients ===")
print("(PHP change in donation revenue per 1-unit increase in feature)")
print()
print(coef_df.to_string(index=False))

# Business interpretation
print()
print("=== Business Interpretation ===")
print(f"Intercept (baseline revenue): ₱{lr_model.intercept_:.2f}")
print()
top3 = coef_df.head(3)
for _, row in top3.iterrows():
    direction = 'INCREASES' if row['coefficient'] > 0 else 'DECREASES'
    print(f"  • {row['feature']}: {direction} donation revenue by ₱{abs(row['coefficient']):.2f} per unit")

print()
print("Prediction vs. Explanation distinction:")
print(f"  Explanatory (LR):  R²={r2_score(y_test, lr_model.predict(X_test_scaled)):.4f} — coefficients are interpretable causal estimates")
print(f"  Predictive  (GB):  R²={results['Gradient Boosting']['R2']:.4f} — black-box but better out-of-sample accuracy")
print()
print("NOTE: LR coefficients show *association*, not causation. High engagement may")
print("cause donations, OR both may be caused by a third factor (e.g., campaign launch).")
print("Treat coefficients as signals for hypothesis testing, not proven causal effects.")

fig, ax = plt.subplots(figsize=(11, 6))
coef_df.head(12).plot(x='feature', y='coefficient', kind='barh', ax=ax, color=[
    'steelblue' if v > 0 else 'tomato' for v in coef_df.head(12)['coefficient']
], legend=False)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Explanatory Model: Linear Regression Coefficients
(blue = positive effect, red = negative effect on donation revenue)')
ax.set_xlabel('Coefficient (₱ change per unit)')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance from best model (Gradient Boosting)
gb_model = models['Gradient Boosting']
gb_model.fit(X_train, y_train)

importance_df = pd.DataFrame({
    'feature': feature_cols + cat_cols,
    'importance': gb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 15 Features Driving Donation Revenue:")
print(importance_df.head(15))

fig, ax = plt.subplots(figsize=(12, 8))
importance_df.head(15).plot(x='feature', y='importance', kind='barh', ax=ax, legend=False)
ax.set_title('Top 15 Features Driving Donation Revenue')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## Causal & Relationship Analysis

### Key Findings

**What content drives donations?**

1. **Engagement metrics matter most** (likes, shares, comments):
   - Posts with high engagement generate higher donation revenue
   - **Implication**: Don't just count likes; track donation conversion (engagement â†’ revenue)

2. **Call-to-action is critical** (has_call_to_action):
   - Posts with explicit "Donate Now" CTAs drive significantly more revenue
   - **Implication**: Every post should have a clear ask

3. **Storytelling works** (features_resident_story):
   - Impact stories (anonymized resident stories) generate higher donations than generic content
   - **Implication**: Invest in storytelling; balance privacy concerns with revenue impact

4. **Timing & frequency matter**:
   - Certain days/times show higher conversion
   - **Implication**: Post strategically; test 2pm Tuesday vs. 10am Sunday

5. **Platform strategy varies**:
   - Facebook may convert better than Instagram (or vice versa)
   - **Implication**: Don't assume one-size-fits-all; allocate resources by conversion, not just follower count

6. **Paid boosting ROI**:
   - If boost_budget is important, paid promotion increases reach & donations
   - **Implication**: Calculate ROI: â‚±X spent on boosting â†’ â‚±Y in donations; aim for 1:5 ratio minimum

### Business Interpretation

**Content Strategy Recommendation:**

- **Post Type**: Prioritize ImpactStory + FundraisingAppeal (not just EducationalContent)
- **Tone**: Use Emotional + Hopeful (not just Informative)
- **Media**: Prefer Video/Reel (higher engagement â†’ higher donations) over Text-only
- **CTA**: Every post should have "DonateNow" or "ShareStory" CTA
- **Stories**: Feature anonymized resident stories 2â€“3x per month
- **Timing**: Test posting on [day/time with highest historical conversion] consistently
- **Platform**: Allocate content by conversion rate, not follower count
- **Boost**: Use paid promotion strategically on high-performing post types; target 1:5 ROI minimum


## Deployment: Content Recommender

**Use in Application:**

1. **Pre-Post Dashboard Widget**:
   - Staff can draft a post, fill in metadata (type, tone, media, CTA)
   - Model predicts estimated donation revenue
   - Recommendation: "This Impact Story with video + DonateNow CTA will likely generate â‚±15â€“25K. Schedule for Tuesday 2pm for best conversion."

2. **Post-Publication Analytics**:
   - After post goes live, track actual vs. predicted donations
   - Dashboard shows "Estimated â‚±20K â†’ Actual â‚±22K âœ“ On target!"

3. **Content Calendar**:
   - Recommended post mix: 30% ImpactStory, 20% Campaign, 20% ThankYou, 20% Education, 10% Event
   - Each with best-performing tone, media type, CTA

4. **A/B Testing**:
   - Test post type A vs. B on small audience; model predicts winner
   - Avoid guessing; use data to decide content strategy


In [ ]:
### Deployment: API Integration Code
# This model is served via GET /api/mlinsights/acquisition-roi
# The Social Media insights are surfaced in the ML Insights dashboard (Pipeline 03 tab).
# Below is a reference snippet showing how the top content recommendations are generated.

print("=== DEPLOYMENT REFERENCE ===")
print("Backend endpoint: GET /api/mlinsights  (social_media section)")
print("Frontend:         ML Insights page → Social tab")
print()

# Score all posts: predict donation revenue
df['predicted_revenue'] = models['Gradient Boosting'].predict(X)
df['explanatory_score'] = lr_model.predict(X_train_scaled[:len(df)] if len(X_train_scaled) >= len(df) else scaler.transform(X))

# Top performing content types (what the dashboard shows)
top_combos = (df.groupby(['post_type', 'sentiment_tone', 'media_type'])
               ['predicted_revenue']
               .mean()
               .sort_values(ascending=False)
               .head(5)
               .reset_index())
print("Top 5 Content Combinations (served to Content Calendar widget):")
print(top_combos.to_string(index=False))

print()
print("Sample API response shape (what /api/mlinsights returns for social section):")
print(json.dumps({
    "topPostTypes": top_combos['post_type'].tolist()[:3],
    "avgRevenueByType": {pt: round(rev, 2) for pt, rev in 
                         df.groupby('post_type')['predicted_revenue'].mean().items()},
    "recommendedTone": df.groupby('sentiment_tone')['predicted_revenue'].mean().idxmax(),
    "ctaLift": round(df[df['has_call_to_action']==1]['estimated_donation_value_php'].mean() - 
                     df[df['has_call_to_action']==0]['estimated_donation_value_php'].mean(), 2)
}, indent=2))

In [ ]:
print("Pipeline 3 complete: Social Media Performance")
print(f"\nKey metric: Donation prediction RMSE â‚±{results['Gradient Boosting']['RMSE']:.2f}")
print(f"Top feature for revenue: {importance_df.iloc[0]['feature']}")